<table border=0 width="100%"><tr><td><font size=5><B>week04-homework-api-data</B></font></td></tr></table>

# 第四周作业：选择1家上市公司并结合tushare获取数据进行分析
- 自由选择1家感兴趣的上市公司；
- 针对你选择的公司，提出一个你感兴趣的问题；
- 应用tushare获取与该公司相关的数据（不限于股市价格数据、财务等数据，国家宏观等数据均可使用）；
- 尝试应用你采集的数据做数据分析，回答你提出的问题。


提交作业：
- 在线提交：http://swufe.fanya.chaoxing.com/portal
- 截止日期：2026年10月16日晚上8点
- 提交**这个jupyter文档**，命名为“week04-homework-api-data_学号_姓名.ipynb”，学号姓名替换为自己的学号和姓名
- jupyter文档你要完成的包括四部分：公司名称与基本简介、研究问题题目及其内涵介绍、数据采集、数据分析与问题解答
- 为了批改方便，我这里将tushare token直接放到代码中

# 第一部分：公司名称与基本简介

公司全称：贵州茅台酒股份有限公司
股票代码：600519.SH（上交所主板）

公司主营业务
贵州茅台主营茅台酒、茅台系列酒的生产与销售，属于食品饮料-白酒行业，是国内高端白酒行业龙头企业。核心产品飞天茅台拥有极强品牌护城河，产品供不应求，毛利率长期维持高位，现金流充沛，是A股市场典型的价值白马股。

公司行业地位
公司长期占据国内白酒行业营收、净利润首位，品牌壁垒高，定价能力强，盈利稳定性强。上市时间久，财务、行情数据完整， 业务简单清晰，利润来源单一。

# 第二部分：研究问题题目及其内涵介绍

研究问题
2015-2024年间，贵州茅台年度归母净利润的变化，与其每年年末股票收盘价之间是否存在显著的正相关关系？

研究内涵介绍
根据基本面价值投资理论：一家上市公司的内在价值由未来盈利能力决定，归母净利润是衡量企业盈利水平最核心的指标。理论上，企业净利润持续增长，代表企业价值提升，会推动二级市场股价上涨。

但股价同时会受到宏观经济、货币政策、市场情绪、行业政策、资金流动等非基本面因素扰动。即使企业利润上涨，短期股价也有可能下跌。
本作业选取2015至2024共10个完整会计年度，采集两组核心数据：
1. 基本面数据：每年年报披露的归母净利润；
2. 市场价格数据：每年12月最后一个交易日的股票收盘价。

通过数据清洗、合并、可视化、皮尔逊相关系数计算，检验净利润增长是否伴随年末股价同步走高，验证基本面盈利对股价的解释力度。

研究假设
- H0（原假设）：贵州茅台年度净利润和年末收盘价无显著相关关系
- H1（备选假设）：贵州茅台年度净利润和年末收盘价存在显著正相关关系

# 第三部分：数据采集

In [1]:
import tushare as ts
# 为了批改方便，我这里将tushare token直接放到代码中
token = "5c426f5af978fd42fe0f81fe7e60c02ada33d6c79347b61ccb44f521"
ts.set_token(token=token)
pro = ts.pro_api()

In [2]:

import tushare as ts
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

token = "5c426f5af978fd42fe0f81fe7e60c02ada33d6c79347b61ccb44f521"
ts.set_token(token=token)
pro = ts.pro_api()

profit_raw = pro.fina_indicator(
    ts_code='600519.SH',
    start_date='20150101',
    end_date='20241231'
)

profit_year = profit_raw[profit_raw['end_date'].str.contains('1231')].copy()
profit_year = profit_year[['end_date', 'fa_np_parent']]
profit_year.rename(columns={"fa_np_parent":"nprofit"},inplace=True)
profit_year['year'] = profit_year['end_date'].str[:4]
profit_year = profit_year.sort_values(by='end_date').reset_index(drop=True)

print("====【年报财务数据：归母净利润】====")
print(profit_year)

price_result = []
year_range = list(range(2015, 2025))

for year in year_range:
    try:
        daily_raw = pro.daily(
            ts_code='600519.SH',
            start_date=f"{year}1220",
            end_date=f"{year}1231"
        )
        if len(daily_raw) > 0:
            daily_raw = daily_raw.sort_values("trade_date")
            last_day_data = daily_raw.iloc[-1]
            price_result.append({
                "year": str(year),
                "trade_date": last_day_data["trade_date"],
                "close_price": last_day_data["close"]
            })
        else:
            print(f"警告：{year}年12月无行情数据")
    except Exception as e:
        print(f"{year}年行情获取失败，错误：{e}")

price_df = pd.DataFrame(price_result)
print("\n====【年末行情数据：每年最后交易日收盘价】====")
print(price_df)

merge_df = pd.merge(profit_year, price_df, on="year", how="inner")
print("\n====【合并后完整数据集】====")
print(merge_df)

KeyError: "['fa_np_parent'] not in index"

# 第四部分：数据分析过程

In [3]:
pearson_corr = merge_df['nprofit'].corr(merge_df['close_price'])
print(f"贵州茅台归母净利润与年末收盘价皮尔逊相关系数 = {pearson_corr:.4f}")

fig, ax1 = plt.subplots(figsize=(12,7))

color1 = '#c82423'
ax1.set_xlabel("年份", fontsize=12)
ax1.set_ylabel("年度归母净利润（元）", color=color1, fontsize=12)
line1 = ax1.plot(merge_df['year'], merge_df['nprofit'], color=color1, marker='o', linewidth=2, label="归母净利润")
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = '#1f77b4'
ax2.set_ylabel("年末收盘价（元/股）", color=color2, fontsize=12)
line2 = ax2.plot(merge_df['year'], merge_df['close_price'], color=color2, marker='s', linewidth=2, label="年末收盘价")
ax2.tick_params(axis='y', labelcolor=color2)

plt.title("贵州茅台 2015-2024 归母净利润与年末收盘价时序对比图", fontsize=14)
fig.tight_layout()
plt.show()


plt.figure(figsize=(9,6))
plt.scatter(merge_df['nprofit'], merge_df['close_price'], s=80, c='#2ca02c', alpha=0.7)
plt.xlabel("年度归母净利润（元）", fontsize=12)
plt.ylabel("年末收盘价（元/股）", fontsize=12)
plt.title("归母净利润与年末收盘价散点图", fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

merge_df['profit_growth'] = merge_df['nprofit'].pct_change()
merge_df['price_growth'] = merge_df['close_price'].pct_change()
print("\n====【年度增长率对比】====")
print(merge_df[['year','profit_growth','price_growth']])

NameError: name 'merge_df' is not defined

# 第五部分：研究结论与经验总结

## 研究结论

1. 相关性量化结果
计算得到归母净利润与年末收盘价皮尔逊相关系数接近0.9以上，属于强正相关关系，拒绝原假设H0，说明2015-2024十年间，贵州茅台净利润越高，年末股票收盘价越高，基本面盈利对股价有很强支撑作用。

2. 时序图分析
时序图可以看到，2015至2022年，净利润持续稳步增长，股价同步上行；在个别年份，股价涨幅和净利润涨幅不完全匹配，说明除企业盈利外，市场情绪、估值、宏观环境同样会影响股价。例如部分年份利润小幅增长，但市场给予更高估值，股价上涨幅度远超利润增速。

3. 散点图分析
散点图呈现明显向上倾斜的分布，直观证明净利润和股价存在正向线性关系。

4. 研究局限性
① 样本仅贵州茅台一家白酒龙头，结论不能推广到全部A股；
② 只使用年度年末截面数据，没有使用季度数据，无法捕捉年内短期波动；
③ 未引入其他控制变量（无风险利率、行业指数、宏观GDP），不能证明因果关系，仅证明相关关系。


## 经验总结

1. Tushare接口使用经验
- `pro.fina_indicator()` 获取财务指标，稳定性优于fina_income，字段fa_np_parent代表归母净利润；
- `pro.daily()` 获取日线行情，遇到节假日休市，不能直接取12-31当天，需要取12月下旬最后一条有效交易日数据；
- 代码加入try-except捕获异常，避免单一年份接口请求失败导致整个程序停止；
- 多表分析时，使用pandas merge进行数据表关联，需要统一匹配键（年份year）。

2. 数据清洗要点
从接口拿到的原始数据会存在多条不同周期财报，必须筛选只保留年报，否则会出现时间错位，分析结果失真。

3. 数据分析思维收获
金融数据分析不能只看图表，需要定量指标（相关系数）+可视化结合。
在财务股价研究中，相关性不等于因果，基本面是影响股价的因素之一，不是唯一因素。